In [1]:
# Instrukcja uruchomienia projektu
#
# Najpierw należy uruchomić komórki notebooka tworzące pliki .py.
# Projekt działa jako kilka równoległych procesów, dlatego każdy plik
# uruchamiamy w osobnym terminalu JupyterLab.
#
# Terminal 1 — Flask API + dashboard:
# cd /home/jovyan/notebooks/Projekt
# python flask_scoring_api.py
#
# Terminal 2 — connector Kafka → Flask API:
# cd /home/jovyan/notebooks/Projekt
# python market_api_connector.py
#
# Terminal 3 — producer danych rynkowych:
# cd /home/jovyan/notebooks/Projekt
# python market_kafka_producer.py
#
# Terminal 4 — opcjonalnie, tylko podgląd danych z Kafki:
# cd /home/jovyan/notebooks/Projekt
# python kafka_market_consumer.py
#
# Główny pipeline:
# producer → Kafka → connector → Flask API → dashboard
#
# Dashboard:
# http://localhost:5000/dashboard
#
# Jeżeli dashboard nie otwiera się w przeglądarce, oznacza to,
# że port 5000 nie jest wystawiony z kontenera Dockera na komputer.
# Wtedy w terminalu systemowym Windows / CMD / PowerShell należy wpisać:
#
# docker inspect -f "{{range $name, $net := .NetworkSettings.Networks}}{{$name}}{{end}}" jupyter
#
# Następnie, po otrzymaniu nazwy sieci, np. jupyterlab-project_default:
#
# docker run --rm --name flask_proxy --network jupyterlab-project_default -p 5000:5000 alpine/socat -dd TCP-LISTEN:5000,fork,reuseaddr TCP:jupyter:5000
#
# Tego terminala nie należy zamykać podczas działania dashboardu,
# ponieważ utrzymuje on przekierowanie portu 5000 do kontenera Jupyter.
#
# Jeśli kontener flask_proxy już istnieje, można go zatrzymać poleceniem:
# docker stop flask_proxy
#
# Po tej komendzie dashboard powinien być dostępny pod adresem:
# http://localhost:5000/dashboard
#
# Zatrzymanie procesów w terminalach: CTRL + C

# 1. Instalacja bibliotek

In [2]:
!pip install yfinance


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
!pip install --user kafka-python


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


# 2. Market producer — pobieranie i wysyłka danych do Kafki

In [4]:
%%file market_kafka_producer.py
import yfinance as yf
from kafka import KafkaProducer
from datetime import datetime
import time
import json
import random

# =========================
# USTAWIENIA
# =========================

TICKERS = ["SPY", "QQQ", "NVDA"]
TOPIC = "market_events"
BOOTSTRAP_SERVERS = "broker:9092"
SLEEP_SECONDS = 30

backup_prices = {
    "SPY": 520.0,
    "QQQ": 450.0,
    "NVDA": 900.0
}

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    value_serializer=lambda v: json.dumps(v, ensure_ascii=False).encode("utf-8")
)


def get_market_event_from_yfinance(ticker):
    """
    Pobiera dane 1-minutowe z yfinance.
    Bierzemy ostatnią świecę z wolumenem > 0, bo najnowsza świeca może być jeszcze niedomknięta.
    """

    try:
        df = yf.Ticker(ticker).history(period="1d", interval="1m")

        if df.empty:
            return None

        df = df.dropna()

        # zostawiamy tylko świece z wolumenem większym niż 0
        df = df[df["Volume"] > 0]

        if len(df) < 2:
            return None

        previous = df.iloc[-2]
        current = df.iloc[-1]

        previous_price = float(previous["Close"])
        current_price = float(current["Close"])

        if previous_price == 0:
            return None

        change_pct = ((current_price - previous_price) / previous_price) * 100

        event = {
            "ticker": ticker,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "price": round(current_price, 4),
            "volume": int(current["Volume"]),
            "change_pct": round(change_pct, 4),
            "hour": datetime.now().hour,
            "source": "yfinance",
            "status": "OK"
        }

        return event

    except Exception as e:
        print(f"Błąd yfinance dla {ticker}: {e}")
        return None


def get_market_event_from_backup(ticker):
    """
    Generator backupowy używany, gdy yfinance nie zwróci poprawnych danych, bo np. giełda jest zamknięta.
    """

    old_price = backup_prices[ticker]

    change_pct = random.choices(
        population=[
            random.uniform(-0.05, 0.05),
            random.uniform(-0.20, 0.20),
            random.uniform(-0.80, 0.80)
        ],
        weights=[80, 15, 5],
        k=1
    )[0]

    new_price = old_price * (1 + change_pct / 100)
    backup_prices[ticker] = new_price

    event = {
        "ticker": ticker,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "price": round(new_price, 4),
        "volume": random.randint(50_000, 5_000_000),
        "change_pct": round(change_pct, 4),
        "hour": datetime.now().hour,
        "source": "backup_generator",
        "status": "OK"
    }

    return event


print("Start Kafka producer dla danych rynkowych.")
print("Topic:", TOPIC)
print("Bootstrap servers:", BOOTSTRAP_SERVERS)
print("Instrumenty:", ", ".join(TICKERS))
print(f"Częstotliwość wysyłki: co {SLEEP_SECONDS} sekund")
print("Zatrzymanie: CTRL + C")
print("-" * 100)

while True:
    for ticker in TICKERS:
        event = get_market_event_from_yfinance(ticker)

        if event is None:
            event = get_market_event_from_backup(ticker)

        producer.send(TOPIC, event)

        print("Wysłano do Kafki:", json.dumps(event, ensure_ascii=False))

    producer.flush()
    print("-" * 100)

    time.sleep(SLEEP_SECONDS)

Overwriting market_kafka_producer.py


In [5]:
# python market_kafka_producer.py 

# 3. Plik zczytujący dane

In [6]:
%%file kafka_market_consumer.py
from kafka import KafkaConsumer
import json

# Konfiguracja konsumenta - najprostsza działająca wersja
consumer = KafkaConsumer(
    'market_events',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)


print("Oczekuję na dane.\n")
print("=" * 80)

# Pętla nasłuchująca na nowe wiadomości
for message in consumer:
    data = message.value
    
    # Wyciąganie danych zgodnie z kluczami zdefiniowanymi przez Dawida w 'event'
    ticker = data.get('ticker', 'UNKNOWN')
    price = data.get('price', 0.0)
    volume = data.get('volume', 0)
    change_pct = data.get('change_pct', 0.0)
    source = data.get('source', 'unknown')
    
    # Eleganckie formatowanie wyjścia dla każdego zdarzenia
    print(f"[{source.upper()}] {ticker} | Cena: {price:.2f} | Wolumen: {volume} | Zmiana: {change_pct:.4f}%")
    
    # Prosty test alertu z Waszego planu (Reguła R1: |change_pct| > 0.05%)
    if change_pct is not None and abs(change_pct) > 0.05:
        print(f"Uwaga: {ticker} - Zmiana {change_pct:.4f}% przekracza dopuszczalny próg 0.05%!")
        
    print("-" * 80)

Overwriting kafka_market_consumer.py


In [7]:
# python kafka_market_consumer.py 

# 4. scoring_service

In [8]:
%%file flask_scoring_api.py

from flask import Flask, request, jsonify
from datetime import datetime

app = Flask(__name__)


# PAMIĘĆ PODRĘCZNA STATYSTYK
stats = {
    "total_scored": 0,
    "by_risk": {"LOW": 0, "MEDIUM": 0, "HIGH": 0, "CRITICAL": 0},
    "by_ticker": {},
    "alerts_generated": 0,
    "started_at": datetime.now().isoformat(timespec="seconds")
}

# Ostatnie zdarzenia do dashboardu
recent_events = []


# LOGIKA SCORINGU
def score_event(data):
    """
    Przyjmuje słownik zdarzenia rynkowego i zwraca score, risk_level oraz listę
    uruchomionych reguł.

    Reguły:
      R1 – umiarkowana zmiana ceny: |change_pct| > 0.05%  → +1 punkt
      R2 – duża zmiana ceny:        |change_pct| > 0.10%  → +2 punkty
      R3 – bardzo duża zmiana ceny: |change_pct| > 0.20%  → +2 punkty
      R4 – podwyższony wolumen LUB źródło backup_generator → +1 punkt

    Progi ryzyka:
      0     → LOW
      1–2   → MEDIUM
      3–4   → HIGH
      5+    → CRITICAL
    """

    change_pct = abs(data.get("change_pct", 0.0))
    volume = data.get("volume", 0)
    source = data.get("source", "yfinance")

    score = 0
    triggered = []

    # R1
    if change_pct > 0.05:
        score += 1
        triggered.append("R1")

    # R2
    if change_pct > 0.10:
        score += 2
        triggered.append("R2")

    # R3
    if change_pct > 0.20:
        score += 2
        triggered.append("R3")

    # R4
    VOLUME_THRESHOLD = 1_000_000
    if volume > VOLUME_THRESHOLD or source == "backup_generator":
        score += 1
        triggered.append("R4")

    # Mapowanie score → risk_level
    if score == 0:
        risk_level = "LOW"
    elif score <= 2:
        risk_level = "MEDIUM"
    elif score <= 4:
        risk_level = "HIGH"
    else:
        risk_level = "CRITICAL"

    return score, risk_level, triggered


# ENDPOINTY

@app.route("/health", methods=["GET"])
def health():
    """Sprawdzenie, czy API działa."""
    return jsonify({
        "status": "ok",
        "timestamp": datetime.now().isoformat(timespec="seconds")
    }), 200


@app.route("/score", methods=["POST"])
def score():
    """
    Przyjmuje JSON zdarzenia rynkowego, ocenia ryzyko i zwraca wynik.
    Dla zdarzeń HIGH i CRITICAL drukuje alert w terminalu.
    """

    data = request.get_json(silent=True)

    # --- walidacja ---
    if not data:
        return jsonify({"error": "Brak danych JSON w żądaniu"}), 400

    required_fields = ["ticker", "timestamp", "price", "volume", "change_pct", "source"]
    missing = [f for f in required_fields if f not in data]

    if missing:
        return jsonify({"error": f"Brakujące pola: {', '.join(missing)}"}), 400

    if not isinstance(data.get("price"), (int, float)) or data["price"] <= 0:
        return jsonify({"error": "Pole 'price' musi być dodatnią liczbą"}), 400

    if not isinstance(data.get("volume"), int) or data["volume"] < 0:
        return jsonify({"error": "Pole 'volume' musi być nieujemną liczbą całkowitą"}), 400

    # --- scoring ---
    score_val, risk_level, triggered_rules = score_event(data)

    # --- aktualizacja statystyk ---
    stats["total_scored"] += 1
    stats["by_risk"][risk_level] += 1

    ticker = data.get("ticker", "UNKNOWN")

    if ticker not in stats["by_ticker"]:
        stats["by_ticker"][ticker] = {"total": 0, "HIGH": 0, "CRITICAL": 0}

    stats["by_ticker"][ticker]["total"] += 1

    if risk_level in ("HIGH", "CRITICAL"):
        stats["by_ticker"][ticker][risk_level] += 1

    # --- alert w terminalu ---
    if risk_level in ("HIGH", "CRITICAL"):
        stats["alerts_generated"] += 1

        print(f"\n{'=' * 60}")
        print(f"  ALERT: {ticker} - {risk_level}")
        print(
            f"  Cena: {data.get('price')} | Zmiana: {data.get('change_pct')}%"
            f" | Wolumen: {data.get('volume')}"
        )
        print(f"  Reguły: {', '.join(triggered_rules)} | Score: {score_val}")
        print(f"  Źródło: {data.get('source')} | {data.get('timestamp')}")
        print(f"{'=' * 60}\n")

    # --- odpowiedź API ---
    response = {
        "ticker": ticker,
        "score": score_val,
        "risk_level": risk_level,
        "triggered_rules": triggered_rules,
        "timestamp": data.get("timestamp")
    }

    # --- zapis ostatnich zdarzeń do dashboardu ---
    recent_events.append({
        "ticker": ticker,
        "timestamp": data.get("timestamp"),
        "price": data.get("price"),
        "volume": data.get("volume"),
        "change_pct": data.get("change_pct"),
        "source": data.get("source"),
        "score": score_val,
        "risk_level": risk_level,
        "triggered_rules": triggered_rules
    })

    # Trzymamy tylko ostatnie 20 zdarzeń
    if len(recent_events) > 20:
        recent_events.pop(0)

    return jsonify(response), 200


@app.route("/stats", methods=["GET"])
def get_stats():
    """Zwraca zagregowane statystyki od uruchomienia API."""
    return jsonify({
        "stats": stats,
        "uptime_since": stats["started_at"],
        "current_time": datetime.now().isoformat(timespec="seconds")
    }), 200


@app.route("/dashboard", methods=["GET"])
def dashboard():
    """Prosty dashboard HTML pokazujący stan systemu i ostatnie zdarzenia."""

    rows = ""

    for event in reversed(recent_events):
        risk = event["risk_level"]

        if risk == "LOW":
            color = "#d4edda"
        elif risk == "MEDIUM":
            color = "#fff3cd"
        elif risk == "HIGH":
            color = "#f8d7da"
        else:
            color = "#f5c6cb"

        rows += f"""
        <tr style="background-color:{color}">
            <td>{event['timestamp']}</td>
            <td>{event['ticker']}</td>
            <td>{event['price']}</td>
            <td>{event['volume']}</td>
            <td>{event['change_pct']}%</td>
            <td>{event['source']}</td>
            <td>{event['score']}</td>
            <td><b>{event['risk_level']}</b></td>
            <td>{', '.join(event['triggered_rules'])}</td>
        </tr>
        """

    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <meta http-equiv="refresh" content="5">
        <title>Market Risk Dashboard</title>

        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 30px;
                background-color: #f4f6f8;
                color: #1f2933;
            }}

            h1 {{
                color: #003049;
                margin-bottom: 25px;
            }}

            h2 {{
                color: #003049;
                margin-top: 30px;
            }}

            .cards {{
                display: flex;
                gap: 15px;
                margin-bottom: 25px;
                flex-wrap: wrap;
            }}

            .card {{
                background: white;
                padding: 18px;
                border-radius: 10px;
                box-shadow: 0 2px 6px rgba(0,0,0,0.12);
                min-width: 150px;
            }}

            .card-title {{
                font-size: 14px;
                color: #555;
            }}

            .number {{
                font-size: 28px;
                font-weight: bold;
                margin-top: 8px;
                color: #003049;
            }}

            table {{
                width: 100%;
                border-collapse: collapse;
                background: white;
                box-shadow: 0 2px 6px rgba(0,0,0,0.12);
            }}

            th, td {{
                padding: 10px;
                border-bottom: 1px solid #ddd;
                text-align: left;
                font-size: 14px;
            }}

            th {{
                background-color: #003049;
                color: white;
            }}

            .footer {{
                margin-top: 20px;
                color: #666;
                font-size: 13px;
            }}
        </style>
    </head>

    <body>
        <h1>Market Risk Monitoring Dashboard</h1>

        <div class="cards">
            <div class="card">
                <div class="card-title">Wszystkie ocenione zdarzenia</div>
                <div class="number">{stats["total_scored"]}</div>
            </div>

            <div class="card">
                <div class="card-title">Alerty HIGH/CRITICAL</div>
                <div class="number">{stats["alerts_generated"]}</div>
            </div>

            <div class="card">
                <div class="card-title">LOW</div>
                <div class="number">{stats["by_risk"]["LOW"]}</div>
            </div>

            <div class="card">
                <div class="card-title">MEDIUM</div>
                <div class="number">{stats["by_risk"]["MEDIUM"]}</div>
            </div>

            <div class="card">
                <div class="card-title">HIGH</div>
                <div class="number">{stats["by_risk"]["HIGH"]}</div>
            </div>

            <div class="card">
                <div class="card-title">CRITICAL</div>
                <div class="number">{stats["by_risk"]["CRITICAL"]}</div>
            </div>
        </div>

        <h2>Ostatnie zdarzenia rynkowe</h2>

        <table>
            <tr>
                <th>Czas</th>
                <th>Ticker</th>
                <th>Cena</th>
                <th>Wolumen</th>
                <th>Zmiana %</th>
                <th>Źródło</th>
                <th>Score</th>
                <th>Ryzyko</th>
                <th>Reguły</th>
            </tr>
            {rows}
        </table>

        <div class="footer">
            Dashboard odświeża się automatycznie co 5 sekund.
            Dane są przechowywane w pamięci działania Flask API.
        </div>
    </body>
    </html>
    """

    return html


# START
if __name__ == "__main__":
    print("Flask API uruchomione.")
    print("Endpointy:")
    print("  GET  /health")
    print("  POST /score")
    print("  GET  /stats")
    print("  GET  /dashboard")
    print("-" * 40)

    app.run(host="0.0.0.0", port=5000, debug=True)

Overwriting flask_scoring_api.py


In [9]:
# python flask_scoring_api.py
#
# Endpointy Flask API:
# http://localhost:5000/health
# http://localhost:5000/stats
# http://localhost:5000/dashboard
#
# Uwaga:
# W Dockerze port 5000 musi być wystawiony na komputer hosta.
# Jeśli dashboard nie otwiera się w przeglądarce, można uruchomić proxy:
#
# docker inspect -f "{{range $name, $net := .NetworkSettings.Networks}}{{$name}}{{end}}" jupyter
#
# docker run --rm --name flask_proxy --network jupyterlab-project_default -p 5000:5000 alpine/socat -dd TCP-LISTEN:5000,fork,reuseaddr TCP:jupyter:5000
#
# Terminala z proxy nie należy zamykać podczas działania dashboardu.

# 5. Procesor danych

In [10]:
%%file market_api_connector.py
import json
import requests
import os
from kafka import KafkaConsumer


# KONFIGURACJA
KAFKA_TOPIC = "market_events"
KAFKA_BROKER = "broker:9092"
FLASK_URL = "http://localhost:5000/score"
TEMP_DIR = "./temp_storage"

# Tworzenie folderu tymczasowego, jeśli nie istnieje
if not os.path.exists(TEMP_DIR):
    os.makedirs(TEMP_DIR)
    print(f"Utworzono folder tymczasowy: {TEMP_DIR}")

# Inicjalizacja konsumenta Kafki
consumer = KafkaConsumer(
    KAFKA_TOPIC,
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset='latest',
    enable_auto_commit=True,
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print(f"Połączono z Kafką. Nasłuchiwanie na topicu: {KAFKA_TOPIC}")
print(f"Dane będą przesyłane do: {FLASK_URL}")
print("-" * 50)

try:
    for message in consumer:
        event = message.value
        
        # 1. Wybór wymaganych pól
        processed_data = {
            "ticker": event.get("ticker"),
            "timestamp": event.get("timestamp"),
            "price": event.get("price"),
            "volume": event.get("volume"),
            "change_pct": event.get("change_pct"),
            "hour": event.get("hour"),
            "source": event.get("source"),
            "status": event.get("status")
        }
        
        print(f"Przetwarzanie zdarzenia dla: {processed_data['ticker']}")

        # 2. Przekazanie danych do Flask API /score
        try:
            response = requests.post(FLASK_URL, json=processed_data, timeout=5)
            
            if response.status_code == 200:
                result = response.json()
                risk = result.get("risk_level", "UNKNOWN")
                score = result.get("score", 0)
                print(f" -> API Score: {score} | Poziom ryzyka: {risk}")
                
                # Opcjonalne: zapis alertów do folderu tymczasowego
                if risk in ["HIGH", "CRITICAL"]:
                    log_path = os.path.join(TEMP_DIR, "alerts_log.txt")
                    with open(log_path, "a") as f:
                        f.write(f"{processed_data['timestamp']} - {processed_data['ticker']} - {risk}\n")
            else:
                print(f" -> Błąd API: {response.status_code}")
                
        except requests.exceptions.ConnectionError:
            print(" -> BŁĄD: Nie można połączyć się z Flask API. Sprawdź, czy flask_scoring_api.py jest uruchomiony.")
        
        print("-" * 50)

except KeyboardInterrupt:
    print("\nZatrzymano procesor.")

Overwriting market_api_connector.py
